## Feature Enrichment and Customer Segmentation

### Objective
Enhance the RFM dataset by incorporating additional behavioral and lifecycle features, including Coupon Ratio and Tenure, in order to improve customer segmentation and clustering performance.

- **Coupon Ratio:** measures how frequently a customer uses discounts.
- **Tenure:** represents the duration of the customer’s relationship with the company.
- **Segmentation:** classify customers based on coupon usage and tenure to capture behavioral patterns.

*Note on Data Selection: An additional e-commerce dataset was initially considered for this phase. However, exploratory data analysis revealed zero overlapping `CustomerID`s with the primary transaction dataset. It was intentionally excluded from this pipeline to maintain data integrity and prevent the introduction of artificial or empty features into the machine learning models.*

### Input Data
- `customer_profile_rfm.csv`: normalized RFM features.
- `Online_Sales_with_VIP.csv`: transaction-level data with VIP labeling.
- `Cleaned_CustomersData.csv`: customer information (including tenure).

### Methodology

1. Compute Coupon Ratio directly from the primary transaction data.
2. Merge customer tenure from the primary customer dataset.
3. Apply rule-based segmentation to capture behavioral patterns:
   - Coupon-based segmentation (Deal-seeker vs Loyal)
   - Tenure-based segmentation
4. Apply log transformation to highly skewed features and standardize the final set to prepare for distance-based clustering.

### Output Data
- **`final_integrated_data_unscaled.csv`**: Unscaled features and segments preserved for exploratory analysis and tree-based machine learning feature engineering.
- **`final_integrated_data.csv`**: The fully transformed and scaled dataset containing RFM, Coupon Ratio, and Tenure, optimized for clustering algorithms.

In [1]:
from mailcap import show

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# load RFM data
rfm_df = pd.read_csv("../data/processed/customer_profile_rfm.csv")

# load additional datasets
sales_df = pd.read_csv("../data/processed/Online_Sales_with_VIP.csv")
customers_df = pd.read_csv("../data/cleaned_data/Cleaned_CustomersData.csv")

# create coupon usage flag
sales_df["Coupon_Status"] = sales_df["Coupon_Status"].fillna("Not Used")
sales_df["Is_Coupon_Used"] = (
    sales_df["Coupon_Status"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("used")
    .astype(int)
)

# compute coupon ratio
coupon_df = (
    sales_df
    .groupby("CustomerID")
    .agg(
        Total_Coupons_Used=("Is_Coupon_Used", "sum"),
        Total_Transactions=("Transaction_ID", "count")
    )
    .reset_index()
)

coupon_df["Coupon_Ratio"] = np.where(
    coupon_df["Total_Transactions"] > 0,
    coupon_df["Total_Coupons_Used"] / coupon_df["Total_Transactions"],
    0
)

# Integrate Features
customers_enriched = customers_df.merge(
    coupon_df[["CustomerID", "Coupon_Ratio"]],
    on="CustomerID",
    how="left"
)

# Ensure consistent naming for Tenure
if "Tenure" in customers_enriched.columns:
    customers_enriched = customers_enriched.rename(columns={"Tenure": "Tenure_Months"})

# Merge into RFM dataset
final_df = rfm_df.merge(
    customers_enriched[["CustomerID", "Coupon_Ratio", "Tenure_Months"]],
    on="CustomerID",
    how="left"
)

# Handle potential missing values after merge
final_df[["Coupon_Ratio", "Tenure_Months"]] = final_df[["Coupon_Ratio", "Tenure_Months"]].fillna(0)

# Rule-Based Segmentation
final_df["Coupon_Segment"] = np.where(
    final_df["Coupon_Ratio"] >= 0.5,
    "Deal-seeker",
    "Loyal"
)

final_df["Tenure_Segment"] = pd.cut(
    final_df["Tenure_Months"],
    bins=[0, 6, 12, 24, np.inf],
    labels=["New", "Short-term", "Mid-term", "Long-term"]
)

# Save unscaled data for future EDA / Tree-based models
final_df.to_csv(
    "../data/processed/final_integrated_data_unscaled.csv",
    index=False
)

In [2]:
# prepare features for clustering
features = final_df[
    ["Recency", "Frequency", "Monetary", "Coupon_Ratio", "Tenure_Months"]
].copy()

cols = features.columns.tolist()

skew_df = pd.DataFrame({
    "Feature": cols,
    "Skewness": [features[col].skew() for col in cols]
})

display(skew_df.sort_values("Skewness", ascending=False))

,Feature,Skewness
2,Monetary,6.747279
1,Frequency,5.689494
3,Coupon_Ratio,0.941275
0,Recency,0.454355
4,Tenure_Months,-0.002652


### Skewness Analysis and Transformation Decision

The skewness analysis indicates clear differences in feature distributions.

Monetary (6.75) and Frequency (5.69) are highly right-skewed, reflecting a long-tail distribution driven by a small number of high-value customers. To reduce the impact of extreme values and improve clustering performance, log transformation is applied to these features.

Coupon_Ratio (0.94) shows moderate skewness but is bounded within [0,1]. It is therefore retained in its original form to preserve interpretability.

Tenure_Months (≈0) exhibits a near-symmetric distribution, requiring no transformation. Recency (0.45) shows only mild skewness and is also kept unchanged.

### Implication for Modeling

Only highly skewed features, specifically Monetary and Frequency, are transformed to reduce the impact of extreme values.

All other features — Recency, Coupon_Ratio, and Tenure_Months — are retained in their original form, as they exhibit stable distributions or are bounded and interpretable by design.

This selective transformation ensures a balanced feature space without distorting meaningful behavioral signals.

In [3]:
# log only skewed features
features["Monetary"] = np.log1p(features["Monetary"])
features["Frequency"] = np.log1p(features["Frequency"])

scaler = StandardScaler()
scaled = scaler.fit_transform(features)

cluster_df = pd.DataFrame(
    scaled,
    columns=features.columns
)

cluster_df.insert(0, "CustomerID", final_df["CustomerID"])

display(cluster_df.head())


,CustomerID,Recency,Frequency,Monetary,Coupon_Ratio,Tenure_Months
0,12346,-0.365961,-1.782571,-1.451163,3.972191,0.364594
1,12347,-0.837001,1.012911,1.486702,-0.035140,-0.423659
2,12348,-0.699614,-0.266074,0.024247,0.836019,0.937869
3,12350,-1.249160,0.023983,-0.040704,0.082723,-0.065362
4,12356,-0.365961,0.179406,0.183935,-0.202112,0.364594


In [4]:
missing = cluster_df.isnull().sum()
missing = missing[missing > 0]

# check if cluster_df is cleaned or not.
if missing.empty:
    print("No missing values")
else:
    percent = (missing / len(cluster_df)) * 100
    result = pd.DataFrame({
        "MissingCount": missing,
        "MissingPercent": percent
    }).sort_values("MissingPercent", ascending=False)
    print(result)

No missing values


## Final Integration Summary

Following the RFM computation, additional features (Coupon Ratio and Tenure) were successfully integrated from a single, consistent customer data source.

By applying targeted rule-based segmentation (e.g., Deal-seeker vs Loyal) prior to mathematical scaling, the dataset maintains high interpretability. This transforms the dataset from a purely transactional summary into a comprehensive behavioral profile, fully prepared for advanced clustering and machine learning modeling.

In [5]:
# Save the finalized, scaled dataset
cluster_df.to_csv(
    "../data/processed/final_integrated_data.csv",
    index=False
)